# Build a Multi-Agent System

**Prerequisites:** `04_orchestration_patterns.ipynb`

One system, out of everything this chapter built: a roster of specialists (nb1), a supervisor that routes to them (nb2), forms in both directions (nb3), a board they work on and an event that fires from it (nb3, nb4).

Then the half that decides whether any of it survives contact with a real day: **what happens when a worker fails, loops, or hands back something that cannot be used.** Every failure in this notebook is one this chapter already produced by accident.

| Where it happened | What went wrong |
|---|---|
| nb2, report B | a specialist ran out of budget and returned *"Stopping: reached the maximum of 8 iterations"*. The supervisor read it as an answer |
| nb2, dynamic delegation | the supervisor asked about "this serial" without the serial, and spent a whole agent run finding out |
| nb3, the board | two agents printed `record(...)` as text instead of calling the tool |
| nb4, the watcher | a condition tested the text for "7731" and missed a risk recorded as `no` |
| nb1, nb2 | answers marked internal, or confident with nothing behind them, sent outward anyway |

**By the end you'll have:**
- One recall desk built from the chapter's parts, handling a real report end to end
- A worker that raises, one that runs out of budget, and one that will not fill in the form — each arriving as a **form the caller can read** rather than as an exception or a sentence
- A gate that decides whether a reply may leave the building, and what it refuses
- A board whose watchers loop, and the guard that stops it
- An honest list of what this system still gets wrong

## Setup

In [1]:
import sys
from pathlib import Path

HERE = Path.cwd().parent
sys.path.append(str(HERE))
for chapter in ("01_foundations", "02_agent_runtime", "03_tools"):
    sys.path.append(str(HERE.parent / chapter))

import json
from typing import Annotated

import aggregate
import failures
import recall
from failures import attempt, guard_watchers, sendable
from dataclasses import replace

from messages import Response, answer, as_form_tool
from meter import Meter
from registry import Registry
from tools import Tool
from state import Shared, on_board
from supervisor import Agent

MODEL = "openai/gpt-oss-120b"        # as notebook 4; notebooks 1-3 measured theirs on 20b

def specialist(name, tools=()):
    return Agent(name, f"{recall.PERSONAS[name]}\n\n{recall.BRIEF}", recall.DESCRIPTIONS[name],
                 tools=Registry(list(tools)), model=MODEL)

support = specialist("support", [recall.lookup_serial])
engineer = specialist("engineer", [recall.cell_lot, recall.test_results])
lawyer = specialist("lawyer")
pr = specialist("pr")
costs = {}

## One system, five patterns

Notebook 4 ended on hybrids: not a category, just one choice per column, repeated wherever the system needs a different answer. A real desk needs several, so this one uses five — each from a different notebook, each doing the job it is good at.

| Part of the desk | Organisation | Control | Shape | From |
|---|---|---|---|---|
| inbound message → a specialist | hierarchical | the supervisor picks | one at a time | nb2 |
| support → engineering, directly | peer-to-peer | the agent that has the question | as needed | nb4 |
| what the case has established | blackboard | — (state, not control) | — | nb3 |
| risk recorded → the lawyer | blackboard | an event on the write | triggered | nb4 |
| "should the recall widen?" | isolated specialists | code, all at once | fan-out → `decisive` | nb3 |
| anything leaving the building | — | a gate in code | — | this notebook |

```
   message ──▶ supervisor ──asks──▶ support ──asks──▶ engineering
                   │                   │                  │
                   │                   └──── board ◀───────┘
                   │                           │
                   │                     risk written
                   │                           ▼
                   │                    event ──▶ lawyer ──▶ regulator_update
                   ▼
              reply (a form)
                   │
              sendable()? ──no──▶ held, with the reason
                   │
                  yes ──▶ out
```

The supervisor is hierarchical and support is peer-to-peer *at the same time*, which is the whole point: the columns are answered per edge, not per system.

In [2]:
SUPERVISOR = """You run the recall desk for the Aria 2 battery recall. Every incoming message comes to you first, and you never write replies yourself.

Decide which specialist should handle each message and ask them, giving them everything they need to know: they see nothing but what you send them. Any report of a unit overheating, swelling or burning goes to engineering first, whatever batch it is from, and before anyone tells a customer their unit is safe.

When you have what you need, call `submit` with the reply that should be sent."""

SUPPORT_JOB = ("\n\nIf a serial is not on the recall list, ask engineering whether the unit could still have "
               "the fault before you tell the customer anything about safety. Record what you establish "
               "on the board.")

REPORT = ("A customer writes: 'My Aria 2 got really hot while charging last night and the case has bulged a bit. "
          "Serial AR2-2409-01544. I'm fine, just worried.'")


def new_board():
    """The case's working state: fields, who owns each one, and a guard on runaway watchers."""
    board = guard_watchers(Shared(
        fields={"serial": "the unit this case is about",
                "in_recall": "whether the serial is on the recall list",
                "cell_lot": "which battery cell lot the unit was built from",
                "risk": "whether this unit could have the recall fault, and what that rests on",
                "regulator_update": "what the regulator has been told since the recall began"},
        owner={"in_recall": "support", "cell_lot": "engineer", "risk": "engineer",
               "regulator_update": "lawyer"},
    ))
    board.set("supervisor", "serial", "AR2-2409-01544")
    return board


def recall_desk():
    """Board, event, peers, supervisor and a gate: five patterns, one desk."""
    board = new_board()
    log, fired = [], []

    # blackboard + event: writing `risk` is what starts the lawyer. Nobody chooses.
    def when_risk_recorded(b, field, value):
        fired.append(attempt(
            on_board(lawyer, b),
            f"Engineering has recorded a risk for unit {b.values.get('serial')}: {value}\n\n"
            "Decide in two sentences whether the regulator must be told anything new, and record it "
            "as regulator_update."))

    board.watch("risk", when_risk_recorded)

    # every specialist works on the board: it reads into their context, and they own their fields
    on_call = on_board(engineer, board)
    at_the_desk = on_board(support, board)

    # peer-to-peer: support holds engineering itself, and its job says when to use it
    at_the_desk = replace(at_the_desk,
                          persona=at_the_desk.persona + SUPPORT_JOB,
                          tools=Registry([*at_the_desk.tools, as_form_tool(on_call, log)]))

    # hierarchical: the supervisor holds the specialists, and knows them only by description
    desk = Agent("supervisor", SUPERVISOR,
                 tools=Registry([as_form_tool(at_the_desk, log), as_form_tool(on_call, log)]),
                 model=MODEL)
    return desk, at_the_desk, board, log, fired


desk, at_the_desk, board, log, fired = recall_desk()
print("supervisor holds:", desk.tools.names)
print("support holds:   ", at_the_desk.tools.names)
print("engineering owns:", [f for f, who in board.owner.items() if who == "engineer"])
print("board fields:    ", list(board.fields))

supervisor holds: ['ask_support', 'ask_engineer']
support holds:    ['lookup_serial', 'record', 'ask_engineer']
engineering owns: ['cell_lot', 'risk']
board fields:     ['serial', 'in_recall', 'cell_lot', 'risk', 'regulator_update']


In [3]:
with Meter() as meter:
    reply = answer(desk, REPORT)
costs["the system"] = meter.summary()

for entry in log:
    print(f"--- asked {entry['agent']}: {entry['request'][:160]}\n{entry['response'].render()[:400]}\n")
for note in fired:
    print(f"--- the event fired the lawyer\n{note.render()[:400]}\n")

print(f"=== the supervisor's reply\n{reply.render()}\n")
print("=== the gate:", sendable(reply))
print("\n=== the board\n" + json.dumps(board.values, indent=1))
print("\n", costs["the system"])

--- asked engineer: Customer reports Aria 2 unit overheating while charging and case bulged. Serial AR2-2409-01544. Determine if unit is affected by battery recall, whether it is f
- Serial AR2-2409-01544 uses battery cells from lot VC-7731 (per supplier data).
- Recall batch is AR2-2408-00001 to AR2-2408-03200; this serial is outside that range → not on recall list.
- Risk: same cell lot as recalled units, which has known separator defect. Engineering rates risk as low but non‑zero; unit could exhibit overheating while charging.
- Safety action: STOP charging immediately. D

--- asked engineer: Unit AR2-2409-01544 is not in the recall batch (AR2-2408) but uses battery cells from lot VC-7731, the same lot as the recalled units. Could this unit still hav
- Serial AR2-2409-01544 is not in the AR2-2408 recall batch → not automatically recalled.
- Battery cells are from lot VC-7731, the same lot identified in the recall.
- Test results for VC-7731: 9 of 11 opened units had a separator defe

### Making the writes happen

Look at the board after that run: `serial` and `in_recall`, nothing else. Engineering answered the question and never called `record`, so `cell_lot` and `risk` were never written — and because `risk` was never written, **the event never fired and the widening decision never ran**. Three of the five patterns were wired up and two of them sat there.

Notebook 3 found the same thing and said what to do about it: reading state is reliable because the code injects it, and writing should be reliable the same way. So the fields the board needs become **fields of the form**, and the code writes them:

```python
class Finding(BaseModel):     # engineering's version of submit
    answer: str
    cell_lot: str             # -> board, by the code
    risk: str                 # -> board, by the code, which fires the event
    confident: bool
    visibility: Literal["internal", "external"]
```

The agent still decides what goes in the fields. What it no longer has to do is remember a second tool call.

In [4]:
from typing import Literal

from pydantic import BaseModel, Field


class Finding(BaseModel):
    """Engineering's form: an answer, plus the two fields the rest of the case needs."""

    answer: str = Field(description="Your assessment, in full, for whoever asked.")
    cell_lot: str = Field(description="The cell lot this unit was built from, exactly as the tool "
                                      "returned it. 'unknown' only if you could not look it up.")
    risk: str = Field(description="One sentence: whether this unit could have the recall fault, "
                                  "and what that rests on.")
    confident: bool = Field(description="True only if your tools support every part of it.")
    visibility: Literal["internal", "external"] = Field(
        description="external if this may be sent to a customer as it stands, internal otherwise.")


def engineering_tool(board, log):
    """Engineering as a tool whose answer lands on the board without anyone remembering to write it."""
    def ask_engineer(request: Annotated[str, "Everything engineering needs: the serial, the symptom, "
                                             "and what you want decided. They see nothing else."]) -> str:
        found = attempt(on_board(engineer, board), request, model=Finding)
        if isinstance(found, Finding):
            # the code writes, so the event behind `risk` cannot be skipped by an agent
            board.set("engineer", "cell_lot", found.cell_lot)
            board.set("engineer", "risk", found.risk)
        log.append({"agent": "engineer", "request": request, "response": found})
        return found.answer if isinstance(found, Finding) else found.render()

    return Tool(ask_engineer, description=recall.DESCRIPTIONS["engineer"])


def recall_desk_v2():
    board, log, fired = new_board(), [], []

    def when_risk_recorded(b, field, value):
        fired.append(attempt(
            on_board(lawyer, b),
            f"Engineering has recorded a risk for unit {b.values.get('serial')}: {value}\n\n"
            "Decide in two sentences whether the regulator must be told anything new, and record it "
            "as regulator_update."))

    board.watch("risk", when_risk_recorded)
    engineering = engineering_tool(board, log)

    at_the_desk = on_board(support, board)
    at_the_desk = replace(at_the_desk, persona=at_the_desk.persona + SUPPORT_JOB,
                          tools=Registry([*at_the_desk.tools, engineering]))
    desk = Agent("supervisor", SUPERVISOR,
                 tools=Registry([as_form_tool(at_the_desk, log), engineering]), model=MODEL)
    return desk, board, log, fired


desk2, board2, log2, fired2 = recall_desk_v2()
with Meter() as meter:
    reply2 = answer(desk2, REPORT)
costs["with the writes in the form"] = meter.summary()

for entry in log2:
    kind = type(entry["response"]).__name__
    print(f"--- asked {entry['agent']} ({kind}): {entry['request'][:120]}")
for note in fired2:
    print(f"\n--- the event fired the lawyer\n{note.render()[:400]}")
print(f"\n=== the reply\n{reply2.render()[:700]}")
print("\n=== the gate:", sendable(reply2))
print("\n=== the board\n" + json.dumps(board2.values, indent=1))
print("\n", costs["with the writes in the form"])

--- asked engineer (Finding): Serial AR2-2409-01544, symptom: unit got hot while charging, case bulged. Determine if this unit is affected by the batt
--- asked engineer (Finding): Serial AR2-2409-01544 uses battery cells from lot VC-7731. Determine if this unit could exhibit the overheating/battery 
--- asked support (Response): Customer serial AR2-2409-01544 reported overheating while charging and case bulging. Engineer determined unit is not in 

--- the event fired the lawyer
The regulator must be informed that engineering has identified a potential risk that unit AR2-2409-01544, which is not on the current recall list, uses VC-7731 cells and could exhibit the recall fault. This information has been recorded as the regulator_update.
[confident: yes; visibility: external]

--- the event fired the lawyer
Yes, the regulator must be informed that unit AR2-2409-01544, although not on the current recall list, uses VC-7731 cells and could exhibit the recall fault. This constitutes a new m

Now the board has a `risk`, so the condition in the next cell is true and the widening decision runs — the fifth pattern, which the first version of the desk skipped without saying so.

### The decision that needs more than one specialist

Routing sends a message to whoever should handle it. Some questions are not like that: *should the recall widen* has an engineering half, a legal half and a public half, and nobody owns it. So the code asks all three at once and decides with `decisive`, which notebook 3 found was the only rule that looked at what an agent had actually checked.

Whether to ask at all is a condition in code, reading the board — the cheapest control there is, and the one that never forgets.

In [5]:
# fan-out, but only when the case calls for it: a condition in code, not an agent's judgement
risk = (board2.values.get("risk") or "").lower()
needs_a_decision = "7731" in risk or "recall fault" in risk or "could" in risk
print("risk on the board:", board2.values.get("risk"))
print("does this need a widening decision?", needs_a_decision)

if needs_a_decision:
    QUESTION = ("Unit AR2-2409-01544 overheated and bulged, and engineering has established what it was "
                "built from. Should the recall be widened to cover batch AR2-2409? Answer yes or no.")
    with Meter() as meter:
        views = aggregate.opinions([engineer, lawyer, pr], QUESTION)
    costs["widen?"] = meter.summary()

    print("\n" + aggregate.render(views))
    print("\ntally:    ", aggregate.tally(views))
    print("majority: ", aggregate.majority(views))
    print("decisive: ", aggregate.decisive(views, "engineer"))
    print("\n", costs["widen?"])

risk on the board: Unit could have the recall fault because it uses VC-7731 cells, but serial is not on the recall list.
does this need a widening decision? True



engineer says no (confident)
  because: Unit AR2-2409-01544 uses the same cell lot VC-7731, but only 2 undetermined cases suggest a low‑risk possibility of wider fault; no confirmed separator defect in batch AR2-2409, so widening recall is not justified yet.
  checked: cell_lot returned VC-7731; test_results shows 9 of 11 opened units had separator defect, 2 undetermined, low but not zero risk for other batches.

lawyer says no (confident)
  because: The incident brief provides no information indicating that batch AR2-2409 is affected; only batch AR2-2408 is identified as having the fault.
  checked: none

pr says no (confident)
  because: The brief only identifies batch AR2-2408 as having the overheating issue; there is no information about batch AR2-2409, so there is no basis to widen the recall.
  checked: brief states affected batch is AR2-2408-00001 to AR2-2408-03200; no mention of AR2-2409.

tally:     {'yes': 0, 'no': 3, 'unsure': 0}
majority:  no
decisive:  ('no', 'engineer ch

## What the desk did, twice

**The first version worked for the customer and quietly skipped two of its own patterns.** The supervisor asked engineering, asked engineering again, then had support write the reply; support told the customer to stop charging and explained that the unit shares the recalled cell lot; the gate passed it (`confident: yes, visibility: external`). 13 calls, 21,632 tokens.

Then look at the board it left behind: `serial` and `in_recall`. No `cell_lot`, no `risk`. Engineering had called `cell_lot`, knew the answer, said it in its reply — and never called `record`. So `risk` was never written, so **the event never fired and the widening decision never ran**. Nothing failed. The system just did three of the five things it was built to do, and said nothing about the other two.

**The second version moves those fields into the form.** `Finding` has `cell_lot` and `risk` next to the answer, and the code writes them when the form comes back. The board ends complete, the event fires the lawyer, `regulator_update` gets written, and the condition on the board is true so the widening decision runs. 18 calls, 27,161 tokens: **26% more for the two patterns that had been silently dormant.**

The reply is also better, and for a reason worth naming: engineering's `risk` sentence was on the board while support wrote, so support's answer leads with "stop using them, stop charging" and then explains the cell lot. In the first version support was working from what the supervisor chose to pass on.

**One rough edge, left in.** The event fired **twice**, because the supervisor asked engineering twice and a watcher fires on every write. The lawyer therefore wrote two nearly identical regulator updates, and the second overwrote the first. Firing only on a *changed* value would not have helped — the two sentences differed slightly. The honest fixes are elsewhere: ask engineering once, or make the consumer idempotent, or have the lawyer read the field it is about to overwrite. A board plus events is not a queue, and it has no idea that two writes were about the same thing.

**And the decision the fan-out produced is defensible**, which is not the same as agreeing with it. All three specialists said not to widen the recall; engineering said so with its tools cited (`cell_lot` → VC-7731, `test_results` → 9 of 11, 2 undetermined, "low but not zero"), while the lawyer cited "none" and PR cited the brief. `majority` and `decisive` agree here, for entirely different reasons — one counted three votes, two of which had checked nothing, and the other read the only evidence in the room.

## Breaking it

Four ways this chapter has already broken, each reproduced deliberately. Three of them need no model at all: a stub stands in for the worker, which is the point — **a failure path you can only test by spending tokens is a failure path nobody tests.**

In [6]:
import llm
from types import SimpleNamespace as NS

def stub(*steps):
    """A model that does exactly what it is told to, in order: ("text", "..."), or ("raise", "...")."""
    remaining = list(steps)

    def create(model=None, messages=None, tools=None, **kw):
        while remaining:
            name, payload = remaining.pop(0)
            if name == "raise":
                raise RuntimeError(payload)
            if name == "text":
                return NS(choices=[NS(message=NS(content=payload, tool_calls=None))],
                          usage=NS(prompt_tokens=10, completion_tokens=1))
        return NS(choices=[NS(message=NS(content="done", tool_calls=None))],
                  usage=NS(prompt_tokens=10, completion_tokens=1))

    return create

real_create = llm.client.chat.completions.create

In [7]:
# 1. the worker raises -- a dead API, a tool that throws, a bug in a tool
llm.client.chat.completions.create = stub(("raise", "connection reset by peer"))
raised = attempt(engineer, "Is AR2-2409-01544 affected?")

# 2. the worker runs out of budget -- nb2's "Stopping: reached the maximum of 8 iterations"
llm.client.chat.completions.create = stub(("text", failures.STOPPED + " of 8 iterations. Last step: "
                                           "decision -> ... This is partial progress, not a final answer."))
ran_out = attempt(engineer, "Is AR2-2409-01544 affected?")

# 3. the worker answers, but not in the form
llm.client.chat.completions.create = stub(("text", "Looks fine to me, probably nothing to worry about."))
no_form = attempt(engineer, "Is AR2-2409-01544 affected?")

llm.client.chat.completions.create = real_create

for name, response in [("raised", raised), ("ran out", ran_out), ("no form", no_form)]:
    print(f"--- {name}\n  answer:     {response.answer[:90]}")
    print(f"  confident:  {response.confident}\n  visibility: {response.visibility}")
    print(f"  unknowns:   {response.unknowns[:90]}\n  sendable:   {sendable(response)}\n")

--- raised
  answer:     Could not do this: engineer raised RuntimeError
  confident:  False
  visibility: internal
  unknowns:   connection reset by peer
  sendable:   (False, 'it is a failure, not an answer: connection reset by peer')

--- ran out
  answer:     Could not do this: engineer ran out of budget before finishing
  confident:  False
  visibility: internal
  unknowns:   Stopping: reached the maximum of 8 iterations. Last step: decision -> ... This is partial 
  sendable:   (False, 'it is a failure, not an answer: Stopping: reached the maximum of 8 iterations. Last step: decision -> ... This i')

--- no form
  answer:     Looks fine to me, probably nothing to worry about.
  confident:  False
  visibility: internal
  unknowns:   This agent answered as text instead of calling submit; nothing in it is checked.
  sendable:   (False, 'the agent is not confident: This agent answered as text instead of calling submit; nothing in it is checked.')



All three arrive as a `Response`, which is the whole point: the caller reads one shape whatever happened.

- **The worker that raised** comes back as *"Could not do this: engineer raised RuntimeError"*, with the original message in `unknowns`. Without this, the exception surfaces inside `Act`, which treats a failure as an argument problem and retries the whole worker through `DebugThink` — three model calls to repair arguments that were never wrong, and another agent run behind each.
- **The worker that ran out of budget** is the notebook 2 failure, caught. `Runtime` ends a run over budget with an ordinary assistant message beginning *"Stopping: reached the maximum…"*, and in notebook 2 the supervisor read that as an answer and carried on. It is now a failure form, and the gate refuses it.
- **The worker that would not use the form** is flagged by `messages.answer` itself: `confident: False`, and `unknowns` saying the agent answered as text so nothing in it is checked. Notice what this is *not* — nobody tried to parse the prose into fields. A form nobody filled in should not look like one that was.

Each of these is a stub standing in for a model. That matters more than it looks: **a failure path you can only exercise by spending tokens is a failure path nobody exercises.** Three of the four failures in this notebook cost nothing to test, so they can run on every change.

### The gate

`sendable` is four `if`s. Each one is a failure this chapter produced with a real model, and each would have been caught here:

In [8]:
CASES = {
    "a good reply": Response(answer="Stop charging and return it at aria.example.com/recall.",
                             confident=True, visibility="external"),
    "internal, sent out": Response(answer="Lot VC-7731 from Voltcell; they have not accepted responsibility.",
                                   confident=True, visibility="internal"),
    "confident, nothing behind it": Response(answer="No other batches are affected.", confident=False,
                                             visibility="external", unknowns="did not check the cell lot"),
    "empty": Response(answer="   ", confident=True, visibility="external"),
    "a failure": failures.failure("engineer ran out of budget before finishing"),
}
for name, response in CASES.items():
    ok, why = sendable(response)
    print(f"{name:<30} {'send' if ok else 'hold':<5} {why}")

a good reply                   send  ok
internal, sent out             hold  marked internal, and the customer is outside the company
confident, nothing behind it   hold  the agent is not confident: did not check the cell lot
empty                          hold  there is no answer in it
a failure                      hold  it is a failure, not an answer: engineer ran out of budget before finishing


Four `if`s, and every one of them is a real failure from earlier in the chapter:

- **internal, sent out** — notebook 1's one agent put the supplier's name and the recall cost into the regulator notice; notebook 2's supervisor forwarded a support answer to a journalist.
- **confident, nothing behind it** — notebook 2's *"no other batches are affected"*, which the database contradicts.
- **empty** — an agent that ran out of tokens mid-answer.
- **a failure** — anything `attempt` caught.

The order matters, and I got it wrong first: a failure is *also* internal and *also* not confident, so the check that fires first decides what the caller is told. Reporting "marked internal" for a dead API sends whoever reads the log looking at the wrong thing.

What the gate cannot do is notice that a confident, external, well-formed answer is **wrong**. `sendable` would have passed notebook 2's "for now, you're safe", because that reply was confident and meant for the customer. Everything the gate checks is a claim the agent made about its own answer. Catching the answer itself being wrong is what the board, the tools and the fan-out are for — and this chapter's evidence is that they help and do not settle it.

### Watchers that feed each other

The event in notebook 4 fires the lawyer when engineering records a risk. Nothing stops the lawyer's write from firing something that writes back.

In [9]:
loop = Shared(fields={"risk": "what engineering found", "regulator_update": "what the regulator was told"})
loop.watch("risk", lambda b, f, v: b.set("lawyer", "regulator_update", f"told them: {v}"))
loop.watch("regulator_update", lambda b, f, v: b.set("engineer", "risk", f"reassessed after: {v}"))

guard_watchers(loop, limit=3)
try:
    loop.set("engineer", "risk", "same cell lot as the recall")
except RuntimeError as stopped:
    print("refused:", stopped)
print("writes that did happen:", [(h["author"], h["field"]) for h in loop.history])

refused: watcher chain deeper than 3 while lawyer was setting regulator_update; something is watching a field it also writes
writes that did happen: [('engineer', 'risk'), ('lawyer', 'regulator_update'), ('engineer', 'risk')]


The guard stops it at depth 3 and says what to look for: *"something is watching a field it also writes."* Two writes did happen before it refused, which is the honest outcome — a guard on the depth cannot undo what the chain already did, so the board is left part-way through and the caller has to deal with that.

This is the cost of events as a control. A manager picking speakers cannot loop like this, because something is choosing each time and can see the transcript. An event has no view of anything: it fires because a field was written, and if what it does writes that field again, it is a while-true. Keep chains one step deep unless you have a reason, and make the guard loud.

## What it cost

In [10]:
print(f"{'':<14}{'calls':>6}{'prompt':>9}{'completion':>12}{'total':>8}{'seconds':>9}")
for name, c in costs.items():
    print(f"{name:<14}{c['calls']:>6}{c['prompt']:>9}{c['completion']:>12}{c['total']:>8}{c['seconds']:>9}")

               calls   prompt  completion   total  seconds
the system        13    17482        4150   21632      108
with the writes in the form    18    22800        4361   27161      149
widen?             5     5084        1333    6417        3


- **The desk costs about 21–27k tokens for one customer report**, 13–18 calls, on gpt-oss-120b. That is the honest price of five patterns: a supervisor call, two engineering runs, a support run, a lawyer run behind the event, and the forms in between.
- **The reliable version costs 26% more than the version that skipped two patterns.** Not because forms are expensive — they are the same call — but because the patterns that had been dormant actually ran.
- **The widening decision costs 5 calls** for three opinions and a verdict, and two of those opinions checked nothing. On this evidence it is worth asking engineering and treating the rest as advice.
- **The failure paths cost nothing.** Three stubs and a guard, no model. That is deliberate: they are the parts you want to run on every change.

## What this system is, and what it still gets wrong

Five patterns, one desk: a supervisor routing (hierarchical), support asking engineering itself (peer-to-peer), a board holding the case (blackboard), an event firing the lawyer (triggered), a fan-out deciding the question nobody owns — and one piece of plain code, the gate, in front of everything that leaves.

**What holds it together is not the agents.** It is the forms between them, the ownership on the board, the condition in code, and the four `if`s in `sendable`. The agents do the reading, the judging and the writing; every guarantee in the system is in the wiring.

**What it still gets wrong**, from its own outputs rather than from imagination:

- **It asked engineering twice** for one report, and fired the event twice, and the lawyer's second update overwrote its first. A board is not a queue and does not know two writes were about the same thing.
- **It cannot tell a wrong answer from a right one.** The gate reads what the agent claims about its answer, not the answer. Notebook 2's "for now, you're safe" would pass every check here.
- **`decisive` is only as good as the agent it names.** It refuses to decide when engineering is unconfident or checked nothing, which is right, and it has nothing to say about engineering being confidently mistaken.
- **Nothing here is measured.** Every live cell in this notebook is one run. Notebook 2 shows what it takes to claim more, and the numbers in notebooks 1–3 come from a different model.
- **Two boards, two model sizes, one scenario.** The chapter's findings are about gpt-oss-20b and 120b on one recall. The method transfers; the numbers do not.

## The chapter, in five lines

1. **A multi-agent system is several agents that are different on purpose** — different personas, different tools. Chapter 2's branches are the same agent again, and a fork is the right answer when one persona can do every part of a job too big for one context.
2. **A supervisor knows an agent by its description**, and routing you depend on belongs in the supervisor's own instructions: measured, 8 of 8 against 0 of 8.
3. **Shape the messages.** A reply with fields can be acted on; a typed request is refused before an agent runs. Self-reported fields are a place to look, enforced ones are a guarantee.
4. **Shared state says where information lives and never who runs next.** That is a separate choice: a manager, an event, or a fixed order.
5. **Failures have to arrive in the same shape as successes**, and the checks that keep a system honest are ordinary code — which is where a multi-agent system is usually thinnest, because the interesting part looks like it is happening inside the agents.